# 01 Default State DRLB (Optuna 10)

May 04 DRLB run with fixed linear lambda init and Optuna(10).

May 04 profiles (`may04_*`) set `lambda_min=float('-inf')` and `lambda_max=float('+inf')` so DRLB does not apply finite λ bounds.


In [8]:
import sys
import json
from dataclasses import replace
from pathlib import Path

import pandas as pd

cwd = Path.cwd().resolve()
repo_root = cwd
while repo_root != repo_root.parent and not (repo_root / 'pyproject.toml').exists():
    repo_root = repo_root.parent
if not (repo_root / 'pyproject.toml').exists():
    raise RuntimeError('Could not locate repository root with pyproject.toml')

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from example_notebooks.experiments.drlb.profiles import build_config as build_drlb_config
from example_notebooks.experiments.drlb.profiles import get_profile as get_drlb_profile
from example_notebooks.experiments.shared_runner import run_experiment_inprocess

import importlib
import simulator.model.drlb.state_representations as drlb_state_representations
import simulator.model.drlb.rl_bid_agent_bat as drlb_rl_bid_agent_bat
import simulator.model.drlb_bidder as drlb_bidder_module

importlib.reload(drlb_state_representations)
importlib.reload(drlb_rl_bid_agent_bat)
importlib.reload(drlb_bidder_module)


<module 'simulator.model.drlb_bidder' from '/Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark/simulator/model/drlb_bidder.py'>

In [9]:
RUN_NAME = 'may04_default_state_optuna10'
DRLB_PROFILE = 'may04_default_linear_lambda_legacy'
VERBOSE = False


In [10]:
config = build_drlb_config(
    run_name=RUN_NAME,
    profile=DRLB_PROFILE,
    split_set='full_train_val_holdout',
)
config = replace(config, n_trials=10)
config = replace(config, refit_on='train_plus_val')
config = replace(config, max_steps=None)

profile_data = get_drlb_profile(DRLB_PROFILE)
base_drlb_params = dict(profile_data['base_drlb_params'])
reference_model_params = dict(profile_data['reference_model_params'])

# Enforce May 04 constraints.
reference_model_params['dqn_gamma'] = 1.0
base_drlb_params['init_lambda'] = 0.0028423174374845716
base_drlb_params['init_lambda_mode'] = 'constant'
base_drlb_params['traffic_path'] = str(repo_root / 'data' / 'traffic_share.csv')

from simulator.model.drlb.state_representations import get_state_repr

state_repr = get_state_repr(profile_data['state_type'])
state_repr.begin_episode(1000.0, total_steps=72)
state_vec_len = len(state_repr.curr_state)
if state_vec_len != state_repr.state_size:
    raise RuntimeError(
        f"State representation mismatch for {profile_data['state_type']}: "
        f"len(curr_state)={state_vec_len}, state_size={state_repr.state_size}. "
        "Rerun from the first cell to refresh imports."
    )

result = run_experiment_inprocess(
    config,
    verbose=VERBOSE,
    base_drlb_params=base_drlb_params,
    reference_model_params=reference_model_params,
    state_type=profile_data['state_type'],
    objective=profile_data['objective'],
    search_space_fn=profile_data['search_space_fn'],
    n_trials=config.n_trials,
    max_train_steps=config.max_steps,
)

summary = result['summary']
summary_path = config.outputs_dir / 'run_summary.json'
print(f'Run summary: {summary_path}')
print(json.dumps({
    'run_name': config.run_name,
    'profile': DRLB_PROFILE,
    'state_type': profile_data['state_type'],
    'init_lambda': base_drlb_params['init_lambda'],
    'init_lambda_mode': base_drlb_params['init_lambda_mode'],
    'n_trials': config.n_trials,
    'tuning_best_params': summary['tuning']['best_params'],
    'best_val_metrics': summary['tuning']['best_val_metrics'],
    'final_holdout_metrics': summary['final_holdout']['metrics'],
    'diagnostics_png': summary['refit']['combined_diagnostics_plot_path'],
}, indent=2))


[I 2026-05-05 00:08:05,720] A new study created in memory with name: no-name-1f56451e-5210-4364-a665-10320b454ba0
[I 2026-05-05 00:11:27,076] Trial 0 finished with value: 2029.023890281349 and parameters: {'dqn_lr': 0.01, 'reward_net_lr': 0.0003, 'bid_lower_clip': 7, 'bid_upper_clip': 6}. Best is trial 0 with value: 2029.023890281349.
[I 2026-05-05 00:15:02,414] Trial 1 finished with value: 1967.4137600196855 and parameters: {'dqn_lr': 0.01, 'reward_net_lr': 0.0001, 'bid_lower_clip': 7, 'bid_upper_clip': 4}. Best is trial 0 with value: 2029.023890281349.
[I 2026-05-05 00:18:29,896] Trial 2 finished with value: 2035.1914021884124 and parameters: {'dqn_lr': 0.01, 'reward_net_lr': 0.0003, 'bid_lower_clip': 1, 'bid_upper_clip': 9}. Best is trial 2 with value: 2035.1914021884124.
[I 2026-05-05 00:21:41,824] Trial 3 finished with value: 2296.6407720908196 and parameters: {'dqn_lr': 0.0001, 'reward_net_lr': 0.0003, 'bid_lower_clip': 6, 'bid_upper_clip': 5}. Best is trial 3 with value: 2296.64

Run summary: /Users/amsafin/code/local_ml/rl/bat-autobidding-benchmark/example_notebooks/experiments/drlb/may04_default_state_optuna10/outputs/run_summary.json
{
  "run_name": "may04_default_state_optuna10",
  "profile": "may04_default_linear_lambda_legacy",
  "state_type": "default",
  "init_lambda": 0.0028423174374845716,
  "init_lambda_mode": "constant",
  "n_trials": 10,
  "tuning_best_params": {
    "dqn_lr": 0.0003,
    "reward_net_lr": 0.01,
    "bid_lower_clip": 3,
    "bid_upper_clip": 8
  },
  "best_val_metrics": {
    "cpc_relative": 623.6409225276884,
    "rmse": 1.3397484786683524,
    "clicks_sum": 2346.8143280230843,
    "quickspend": 0.027237354085603113,
    "skipped_campaigns": 0,
    "time_inference_sec": 17.65835690498352,
    "time_overall_sec": 22.35964608192444,
    "average_end_balance_share": 0.6489168626714125,
    "label": "best_val",
    "train_steps": 48240,
    "last_dqn_loss": 22.604793548583984,
    "last_reward_net_loss": 484.7668151855469,
    "dqn_los

In [11]:
rows = [
    {'artifact': 'run_summary_json', 'path': str(config.outputs_dir / 'run_summary.json')},
    {'artifact': 'metrics_json', 'path': str(config.outputs_dir / 'metrics.json')},
    {'artifact': 'drlb_diagnostics_png', 'path': str(config.outputs_dir / 'drlb_diagnostics.png')},
    {'artifact': 'best_refit_model', 'path': str(config.best_models_dir / 'best_refit.pt')},
]
pd.DataFrame(rows)


,artifact,path
0,run_summary_json,/Users/amsafin/code/local_ml/rl/bat-autobiddin...
1,metrics_json,/Users/amsafin/code/local_ml/rl/bat-autobiddin...
2,drlb_diagnostics_png,/Users/amsafin/code/local_ml/rl/bat-autobiddin...
3,best_refit_model,/Users/amsafin/code/local_ml/rl/bat-autobiddin...
